# Phase 3: Feature Engineering

This notebook demonstrates the feature engineering pipeline:
- Sequence features (order, recency, frequency)
- Job features (text similarity with NLP)
- Session features (behavior patterns)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import ast
from pathlib import Path

from src.pipeline.feature_engineer import FeatureEngineer

sns.set_style('whitegrid')
%matplotlib inline

## Load Data

In [ ]:
# Load data
data_path = Path('../Data')

x_train = pd.read_csv(data_path / 'x_train_Meacfjr.csv')
y_train = pd.read_csv(data_path / 'y_train_SwJNMSu.csv')
x_test = pd.read_csv(data_path / 'x_test_jCBBNP2.csv')

# Load job listings
with open(data_path / 'job_listings.json', 'r', encoding='utf-8') as f:
    jobs = json.load(f)

print(f"Train sessions: {len(x_train)}")
print(f"Test sessions: {len(x_test)}")
print(f"Job listings: {len(jobs)}")

## Prepare Sequences

In [ ]:
def parse_sequence(sequence_str):
    """Parse string representation of list to actual list"""
    try:
        return ast.literal_eval(sequence_str)
    except:
        return []

# Parse sequences
x_train['jobs_list'] = x_train['job_ids'].apply(parse_sequence)
x_train['actions_list'] = x_train['actions'].apply(parse_sequence)

x_test['jobs_list'] = x_test['job_ids'].apply(parse_sequence)
x_test['actions_list'] = x_test['actions'].apply(parse_sequence)

print("Sample session:")
print(f"Jobs: {x_train.iloc[0]['jobs_list']}")
print(f"Actions: {x_train.iloc[0]['actions_list']}")

## Initialize Feature Engineer

In [ ]:
# Initialize feature engineer with job listings
feature_engineer = FeatureEngineer(jobs_dict=jobs)

print("Feature Engineer initialized successfully!")

## Extract Sequence Features

In [ ]:
# Extract sequence features only (faster for initial exploration)
x_train_seq = feature_engineer.add_sequence_features(x_train.head(100))

print("\nSequence Features:")
sequence_features = ['unique_jobs_count', 'repeat_rate', 'consecutive_views', 
                     'consecutive_applies', 'view_to_apply_transitions']
print(x_train_seq[sequence_features].head())

# Visualize sequence features
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(x_train_seq['unique_jobs_count'], bins=30, edgecolor='black')
axes[0, 0].set_title('Unique Jobs per Session')
axes[0, 0].set_xlabel('Count')

axes[0, 1].hist(x_train_seq['repeat_rate'], bins=30, edgecolor='black')
axes[0, 1].set_title('Repeat Rate')
axes[0, 1].set_xlabel('Rate')

axes[1, 0].hist(x_train_seq['consecutive_views'], bins=30, edgecolor='black')
axes[1, 0].set_title('Max Consecutive Views')
axes[1, 0].set_xlabel('Count')

axes[1, 1].hist(x_train_seq['view_to_apply_transitions'], bins=30, edgecolor='black')
axes[1, 1].set_title('View to Apply Transitions')
axes[1, 1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## Extract Session Features

In [ ]:
# Add session features
x_train_session = feature_engineer.add_session_features(x_train_seq)

print("\nSession Features:")
session_features = ['session_length', 'view_ratio', 'apply_ratio', 
                    'session_diversity', 'exploration_score']
print(x_train_session[session_features].head())

# Visualize session features
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(x_train_session['view_ratio'], bins=30, edgecolor='black')
axes[0, 0].set_title('View Ratio Distribution')
axes[0, 0].set_xlabel('Ratio')

axes[0, 1].hist(x_train_session['apply_ratio'], bins=30, edgecolor='black')
axes[0, 1].set_title('Apply Ratio Distribution')
axes[0, 1].set_xlabel('Ratio')

axes[1, 0].hist(x_train_session['session_diversity'], bins=30, edgecolor='black')
axes[1, 0].set_title('Session Diversity')
axes[1, 0].set_xlabel('Diversity Score')

axes[1, 1].scatter(x_train_session['exploration_score'], 
                   x_train_session['exploitation_score'], alpha=0.5)
axes[1, 1].set_title('Exploration vs Exploitation')
axes[1, 1].set_xlabel('Exploration Score')
axes[1, 1].set_ylabel('Exploitation Score')

plt.tight_layout()
plt.show()

## Extract All Features (Including NLP Job Features)

In [ ]:
# Extract all features (this may take a few minutes for the full dataset)
print("Extracting all features (including NLP job features)...")
print("This may take a few minutes...\n")

# Start with a small sample
sample_size = 500
x_train_sample = x_train.head(sample_size).copy()

# Extract all features
x_train_features = feature_engineer.extract_all_features(x_train_sample)

print(f"\nExtracted {len(x_train_features.columns)} columns")
print("\nFeature columns:")
print(feature_engineer.get_feature_names())

## Analyze Job Similarity Features

In [ ]:
# Visualize job similarity features
job_features = ['avg_job_similarity', 'max_job_similarity', 'min_job_similarity', 
                'job_title_diversity', 'unique_skills_count']

print("\nJob Similarity Features:")
print(x_train_features[job_features].describe())

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(x_train_features['avg_job_similarity'].dropna(), bins=30, edgecolor='black')
axes[0, 0].set_title('Average Job Similarity')
axes[0, 0].set_xlabel('Similarity Score')

axes[0, 1].hist(x_train_features['job_title_diversity'].dropna(), bins=30, edgecolor='black')
axes[0, 1].set_title('Job Title Diversity')
axes[0, 1].set_xlabel('Diversity Score')

axes[1, 0].hist(x_train_features['unique_skills_count'].dropna(), bins=30, edgecolor='black')
axes[1, 0].set_title('Unique Skills Count')
axes[1, 0].set_xlabel('Count')

axes[1, 1].scatter(x_train_features['avg_job_similarity'], 
                   x_train_features['session_diversity'], alpha=0.5)
axes[1, 1].set_title('Job Similarity vs Session Diversity')
axes[1, 1].set_xlabel('Job Similarity')
axes[1, 1].set_ylabel('Session Diversity')

plt.tight_layout()
plt.show()

## Feature Correlation Analysis

In [ ]:
# Select numeric features for correlation analysis
numeric_features = feature_engineer.get_feature_names()
correlation_data = x_train_features[numeric_features].select_dtypes(include=[np.number])

# Calculate correlation matrix
correlation_matrix = correlation_data.corr()

# Plot correlation heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Save Engineered Features

In [ ]:
# Process full training set
print("Processing full training set...")
x_train_full = feature_engineer.extract_all_features(x_train)

# Process test set
print("Processing test set...")
x_test_full = feature_engineer.extract_all_features(x_test)

# Save features
output_path = Path('../Data')
x_train_full.to_csv(output_path / 'x_train_features.csv', index=False)
x_test_full.to_csv(output_path / 'x_test_features.csv', index=False)

print("\nFeatures saved successfully!")
print(f"Training features shape: {x_train_full.shape}")
print(f"Test features shape: {x_test_full.shape}")

## Summary

We have successfully implemented Phase 3 Feature Engineering:

### 1. Sequence Features
- **Order**: Position of jobs in the sequence (first, last, middle)
- **Recency**: Weighted scores based on position from end (exponential decay)
- **Frequency**: Count of job appearances, repeat rate
- **Patterns**: Consecutive actions, transitions between view/apply

### 2. Job Features (NLP)
- **Text Similarity**: Cosine similarity between job descriptions using TF-IDF
- **Content Features**: Text length, title diversity
- **Skill Analysis**: Unique skills, common keywords across jobs

### 3. Session Features
- **Behavior Patterns**: View/apply ratios, session length
- **Diversity**: Unique jobs vs total jobs in session
- **Exploration vs Exploitation**: Balance between browsing and focusing
- **Action Sequences**: Patterns in how users browse (starts with view, ends with apply)

These features capture:
- User intent and engagement patterns
- Job content relationships
- Sequential behavior patterns
- User exploration strategies